# SFT 实验笔记与执行记录

这个 notebook 用来记录 `base -> SFT -> evaluation` 阶段的学习笔记、实验命令、数据检查、训练记录和结果对比。当前阶段的目标不是训练通用助手，而是先用 0.5B/0.6B 级别 base model 跑通一个可验证的 GSM8K SFT 闭环。

建议每次实验都记录：模型路径、数据路径、训练方式、关键超参、训练日志、固定 prompt 输出、验证集指标，以及你对现象的解释。

## 模块 0. SFT 机制笔记：为什么 GSM8K SFT 能让 base model 更像在回答问题

### 0.1 核心结论

LLM 在 SFT 后并没有摆脱“续写”机制。它仍然是在给定前文 tokens 的条件下预测下一个 token。区别是：SFT 数据和推理输入都被包装成了模型训练时兼容的对话模板，因此模型续写的位置被明确放在 `assistant` 轮次上。模型学到的是：当上下文中出现 `user` 提问并进入 `assistant` 起始位置时，后续应该生成回答，而不是继续写用户的话。

所以更准确的理解是：

```text
base model: 会续写文本，但不稳定知道自己应该扮演 assistant
SFT model : 仍然续写文本，但更倾向在 assistant 位置生成符合训练数据风格的回答
```

### 0.2 GSM8K 原始数据是什么

GSM8K 是小学数学应用题数据集。每条样本通常包含：

```text
question: 自然语言数学应用题
answer  : 解题过程 + 最终答案，最终答案一般放在 #### 后面
```

示意样本：

```text
question:
There are 5 boxes. Each box has 8 apples. How many apples are there?

answer:
There are 5 * 8 = 40 apples.
#### 40
```

它适合入门 SFT，不是因为它能教通用聊天，而是因为它有明确、可验证的任务目标：读题、推理、计算、按固定格式输出最终答案。

### 0.3 verl 如何把 GSM8K 变成 SFT 数据

本仓库的 `examples/data_preprocess/gsm8k_multiturn_sft.py` 会把原始 QA 样本转成 `messages`：

```python
messages = [
    {
        "role": "user",
        "content": question + ' Let\'s think step by step and output the final answer after "####".',
    },
    {
        "role": "assistant",
        "content": answer_raw,
    },
]
```

也就是说，GSM8K SFT 的训练样本不是裸文本，而是：

```text
user:
数学题 + 格式指令

assistant:
解题过程 + #### 最终答案
```

### 0.4 messages 不是模型天然理解的对象

`messages` 是数据结构，不是模型真正直接看到的东西。训练或推理时，tokenizer 会通过 chat template 把它展开成一串 token。例如 Qwen 风格可能类似：

```text
<|im_start|>user
题目 ... Let's think step by step ... "####".<|im_end|>
<|im_start|>assistant
解题过程
#### 答案<|im_end|>
```

模型实际学习的是这些 token 序列中的条件概率，而不是 JSON 对象本身。

### 0.5 SFT 训练时到底在优化什么

因果语言模型的训练目标仍然是 next-token prediction。给定 `user` 内容和已经生成的前缀，模型逐 token 预测 assistant 答案：

```text
最大化：
P(assistant_1 | user)
* P(assistant_2 | user, assistant_1)
* P(assistant_3 | user, assistant_1, assistant_2)
* ...
```

对应到 loss，可以理解成：

```text
loss = -log P(assistant tokens | user tokens)
```

在 verl 的 `MultiTurnSFTDataset` 中，`user` 部分的 loss mask 是 0，`assistant` 部分的 loss mask 是 1。因此模型不是在学习背诵用户问题，而是在学习：给定用户问题后，如何生成 assistant 回答。

### 0.6 为什么这会让 base model 更像在回答问题

base model 预训练时主要学习的是通用文本续写。用户裸输入一个问题时，base model 可能回答，也可能继续写文章、续写对话、输出另一个问题，行为不稳定。

SFT 之后，模型参数或 LoRA adapter 被更新，它在训练中反复看到：

```text
<user> 提问 </user>
<assistant> 回答 </assistant>
```

于是它在类似上下文中会更倾向续写成 assistant 的回答。这就是“从续写变得像回答”的核心原因。

但要注意：它不是机制上不再续写，而是训练分布把续写方向校准到了回答模式。

### 0.7 为什么推理时也要用 chat template

正常 chat LLM 部署时，用户虽然输入格式千奇百怪，但服务端不会把裸文本直接送给模型。服务端通常会包装成模型训练时兼容的模板：

```text
<|im_start|>system
你是一个有帮助的助手。<|im_end|>
<|im_start|>user
用户问题<|im_end|>
<|im_start|>assistant
```

模型真正续写的位置在最后的 `assistant` 起点后面。因此它不是在裸问题后随意续写，而是在明确的 assistant 轮次里生成回答。

这也是为什么训练模板和推理模板要尽量一致。如果训练时用 Qwen chat template，推理时也应该用同一个 tokenizer 的 `apply_chat_template`。

### 0.8 如果推理时不追加 GSM8K 的格式指令会怎样

GSM8K SFT 的 user 内容里追加了：

```text
Let's think step by step and output the final answer after "####".
```

因此最稳定的推理输入也应该保留类似格式指令。三种情况的预期不同：

```text
数学题 + 同样格式指令
=> 最稳定输出 step-by-step + ####

数学题 + 无格式指令
=> 可能仍输出 step-by-step + ####，但格式遵循率可能下降

普通问题 + 无格式指令
=> 不应该期待稳定 ####；如果普通问题也总是 ####，说明 GSM8K 风格偏置过强
```

所以，SFT 后模型会更倾向 GSM8K 风格，但不保证在没有格式指令时仍然稳定输出 `####`。

### 0.9 如果换成其他模板是否也可以

可以，但前提是训练和推理一致。LLM 可以被看成“模板条件下的续写器”，但模板不是魔法。稳定行为来自三件事一起作用：

```text
训练数据分布
+ 推理 chat/template 一致
+ 模型原本已有的语言/知识/推理能力
```

如果你想让模型做 JSON 抽取器，可以在 SFT 阶段训练：

```text
user: 输入一段文本，要求抽取字段
assistant: {"field": "value"}
```

如果你想让模型做判分器，可以训练：

```text
user: prompt + response + rubric
assistant: score + reason
```

关键是：训练时是什么模板，推理时就尽量是什么模板。

### 0.10 当前实验应该验证什么

不要只看 train loss。至少固定两类 prompt 对比 `base` 和 `sft`：

```text
A. GSM8K/math prompts
- 是否输出 ####
- final answer 是否正确
- response 是否过长

B. general QA prompts
- 是否仍能正常回答普通问题
- 是否错误套用 ####
- 是否变得过度 step-by-step
```

推荐记录表：

```text
checkpoint | GSM8K exact match | format rate | avg length | general QA 是否乱套 ####
base       | ...               | ...         | ...        | ...
sft        | ...               | ...         | ...        | ...
```

健康现象：GSM8K 格式遵循率上升，数学输出更稳定，普通问答没有明显被污染。

风险信号：普通问答也强行输出 `####`，所有问题都变成长篇推理，训练 loss 降低但验证正确率不升，或者输出长度明显变长。

## 模块 1. 实验元信息

每次正式训练前先填写这一块。后面复盘时，不要只记“跑过了”，要能追溯到模型、数据、命令和日志。

```text
实验编号:
日期:
base model:
SFT 方法: LoRA / QLoRA / verl SFT trainer / TRL SFTTrainer
训练数据:
验证数据:
训练样本数:
max length:
learning rate:
batch size:
gradient accumulation:
epoch:
输出目录:
主要观察:
下一步动作:
```


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json

os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"

YHY_DIR = Path(r"D:\learnAI\verl\yhy")
VERL_ROOT = YHY_DIR.parent
SFT_DIR = YHY_DIR / "scripts" / "sft"
BASE_MODEL_DIR = YHY_DIR / "models" / "base" / "qwen3_0d6B"
GSM8K_SFT_DIR = YHY_DIR / "datasets" / "gsm8k_sft"
OUTPUT_DIR = YHY_DIR / "models" / "sft"
BASE_EVAL_OUTPUT_DIR = YHY_DIR / "eval_results" / "base_model"
SFT_EVAL_OUTPUT_DIR = YHY_DIR / "eval_results" / "sft_model"

print("python:", sys.version)
print("YHY_DIR:", YHY_DIR)
print("VERL_ROOT:", VERL_ROOT)
print("BASE_MODEL_DIR exists:", BASE_MODEL_DIR.exists())
print("GSM8K_SFT_DIR:", GSM8K_SFT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("BASE_EVAL_OUTPUT_DIR:", BASE_EVAL_OUTPUT_DIR)
print("SFT_EVAL_OUTPUT_DIR:", SFT_EVAL_OUTPUT_DIR)


## 模块 2. 环境检查

目标：确认当前 kernel 能加载 PyTorch、Transformers，并且能看到 CUDA。这里不训练，只做最小环境检查。

In [ ]:
import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))


## 模块 3. 生成 GSM8K SFT 数据

这一节会调用 verl 的预处理脚本，把 `openai/gsm8k` 转成 SFT 使用的 `messages` parquet。第一次运行需要能够访问 Hugging Face；如果网络不可用，需要先离线下载或设置 `--local_dataset_path`。

In [ ]:
GSM8K_SFT_DIR.mkdir(parents=True, exist_ok=True)

# 之前这里使用 subprocess.run(cmd, cwd=VERL_ROOT, check=True) 调用 verl 的预处理脚本：
#   D:\learnAI\verl\examples\data_preprocess\gsm8k_multiturn_sft.py
# 但当前 test3 环境运行该脚本会先遇到两个环境问题：
# 1. 直接用绝对路径执行脚本时，Python 没有自动把 D:\learnAI\verl 加入 import 搜索路径，导致找不到本地 verl 包。
# 2. 加上 PYTHONPATH 后，脚本 import verl.utils.hdfs_io 会触发 verl.__init__，进而 import ray；当前环境没有安装 ray。
#
# 注意：这不是 GSM8K 下载失败。脚本在执行到 datasets.load_dataset("openai/gsm8k", "main") 之前就已经 import 失败。
#
# 为了先完成 SFT 数据准备，这里直接复刻 gsm8k_multiturn_sft.py 的核心逻辑：
# - 用 Hugging Face datasets 下载 openai/gsm8k 的 main 配置。
# - 读取 train/test split。
# - 把 question/answer 转成 messages。
# - 保存为 train.parquet 和 test.parquet。
#
# datasets.load_dataset 的行为：
# - 先检查本地 Hugging Face cache 是否已有 openai/gsm8k。
# - 如果没有缓存，就从 Hugging Face Hub 下载。
# - 返回 DatasetDict，其中包含 dataset["train"] 和 dataset["test"]。

from datasets import load_dataset

instruction_following = 'Let\'s think step by step and output the final answer after "####".'

def to_sft_messages(example):
    question = example["question"] + " " + instruction_following
    answer = example["answer"]
    return {
        "messages": [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
    }

dataset = load_dataset("openai/gsm8k", "main")
train_dataset = dataset["train"].map(to_sft_messages, remove_columns=dataset["train"].column_names)
test_dataset = dataset["test"].map(to_sft_messages, remove_columns=dataset["test"].column_names)

train_path = GSM8K_SFT_DIR / "train.parquet"
test_path = GSM8K_SFT_DIR / "test.parquet"

train_dataset.to_parquet(str(train_path))
test_dataset.to_parquet(str(test_path))

print("saved:", train_path)
print("saved:", test_path)
print("train rows:", len(train_dataset))
print("test rows:", len(test_dataset))


## 模块 4. 检查 SFT 数据格式

目标：确认 parquet 中有 `messages` 列，并观察第一条样本是否是 `user -> assistant` 格式。

In [ ]:
import pandas as pd

train_file = GSM8K_SFT_DIR / "train.parquet"
test_file = GSM8K_SFT_DIR / "test.parquet"

print("train exists:", train_file.exists(), train_file)
print("test exists:", test_file.exists(), test_file)

if train_file.exists():
    df = pd.read_parquet(train_file)
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    print("first messages:")
    print(df.iloc[0]["messages"])
else:
    print("还没有生成 train.parquet。先运行上一节的预处理命令。")


## 模块 5. 抽取小样本

第一次 SFT 不要直接跑完整 GSM8K。先抽 500 或 1000 条，目标是验证数据、模板、loss、显存和保存流程是否正常。

In [ ]:
# 模块 5：从完整 GSM8K SFT 训练集里抽取一个小样本训练集。
#
# 为什么要先抽小样本？
# - 第一次 SFT 的目标不是追求效果，而是验证训练链路能不能跑通。
# - 小样本可以更快暴露数据格式、chat template、loss mask、显存和保存 checkpoint 的问题。
# - 如果 500 条都跑不通，直接跑完整 train.parquet 只会更难排错。
#
# 注意：这个文件只用于训练冒烟测试，不用于评估最终效果。
# 真正评估应该使用 test.parquet 或从 test.parquet 抽出的 eval_20.parquet。

# N 控制抽取多少条训练样本。
# RTX 4070 + 0.6B 模型的第一轮 LoRA SFT，建议先从 500 条开始。
# 如果训练流程稳定，再逐步扩大到 1000、2000 或完整训练集。
N = 500

# small_train_file 是抽样后保存的小训练集路径。
# 例如 N=500 时，会保存为：
#   D:\learnAI\verl\yhy\datasets\gsm8k_sft\train_500.parquet
small_train_file = GSM8K_SFT_DIR / f"train_{N}.parquet"

# train_file 在模块 4 中定义，指向完整的 GSM8K SFT 训练集：
#   D:\learnAI\verl\yhy\datasets\gsm8k_sft\train.parquet
# 这里先检查它是否存在，避免文件路径错误时直接报一长串异常。
if train_file.exists():
    # 读取完整训练集。parquet 是列式二进制格式，不能直接用文本编辑器查看，
    # 需要用 pandas / datasets 这类库读取。
    df = pd.read_parquet(train_file)

    # head(N) 表示取前 N 条样本。
    # 对第一轮冒烟实验来说，顺序抽样足够简单、可复现。
    # 如果后面要做更严谨的小样本实验，可以改成 sample(N, random_state=固定种子)。
    df_small = df.head(N).copy()

    # 把小样本继续保存成 parquet。
    # 这样后面的训练脚本可以像读取完整训练集一样读取它，
    # 不需要额外处理 DataFrame 或 notebook 内存变量。
    df_small.to_parquet(small_train_file)

    # 打印保存路径和行数，作为这一步成功的最小证据。
    print("saved:", small_train_file)
    print("rows:", len(pd.read_parquet(small_train_file)))
else:
    # 如果走到这里，说明模块 3 的数据生成还没成功，或者路径变量被改错了。
    # 先回到模块 3/4，确认 train.parquet 是否存在且能被读取。
    print("train.parquet 不存在，暂时不能抽样。请先运行模块 3 生成数据，并运行模块 4 检查数据格式。")


## 模块 5.1 查看 train_500.parquet 内容

parquet 是列式二进制格式，不能像 txt/json 那样直接用文本编辑器查看。这里用 pandas 读取并检查它是否满足 SFT 训练要求：每行必须有两段 `messages`，第一段是 `user`，第二段是 `assistant`，并且 user 中有格式指令、assistant 中有 `####` 最终答案。


In [ ]:
small_train_file = GSM8K_SFT_DIR / "train_500.parquet"
df_small = pd.read_parquet(small_train_file)

print("file:", small_train_file)
print("shape:", df_small.shape)
print("columns:", df_small.columns.tolist())
print("messages type:", type(df_small.iloc[0]["messages"]))

bad_rows = []
for i, msgs in enumerate(df_small["messages"]):
    try:
        ok = (
            len(msgs) == 2
            and msgs[0]["role"] == "user"
            and msgs[1]["role"] == "assistant"
            and bool(msgs[0]["content"])
            and bool(msgs[1]["content"])
        )
        if not ok:
            bad_rows.append(i)
    except Exception:
        bad_rows.append(i)

user_len = df_small["messages"].map(lambda m: len(m[0]["content"]))
assistant_len = df_small["messages"].map(lambda m: len(m[1]["content"]))
format_instruction_count = df_small["messages"].map(lambda m: "####" in m[0]["content"]).sum()
assistant_marker_count = df_small["messages"].map(lambda m: "####" in m[1]["content"]).sum()

print("role/content ok rows:", len(df_small) - len(bad_rows), "/", len(df_small))
print("bad rows:", bad_rows[:10])
print("format instruction count:", format_instruction_count, "/", len(df_small))
print("assistant #### count:", assistant_marker_count, "/", len(df_small))
print("user char len min/mean/max:", int(user_len.min()), round(float(user_len.mean()), 1), int(user_len.max()))
print("assistant char len min/mean/max:", int(assistant_len.min()), round(float(assistant_len.mean()), 1), int(assistant_len.max()))

print("\nfirst sample:")
print(df_small.iloc[0]["messages"])

print("\nlast sample:")
print(df_small.iloc[-1]["messages"])


## 模块 6. 推理模板实验设计

SFT 前后都用同一批 prompt 对比。至少测试三种输入：

```text
1. 裸数学题
2. 数学题 + GSM8K 格式指令
3. chat template 包装后的数学题 + GSM8K 格式指令
```

重点观察：是否输出 `####`、最终答案是否正确、response 是否过长、普通问答是否被 GSM8K 风格污染。

In [ ]:
FIXED_PROMPTS = {
    "math_plain": "There are 7 bags. Each bag has 6 candies. How many candies are there?",
    "math_with_format": "There are 7 bags. Each bag has 6 candies. How many candies are there? Let's think step by step and output the final answer after \"####\".",
    "general_qa": "请用三句话解释什么是监督微调 SFT。",
    "translation": "Translate this sentence into Chinese: The model learns to answer after supervised fine-tuning.",
}

for name, prompt in FIXED_PROMPTS.items():
    print("====", name, "====")
    print(prompt)


## 模块 6.1 Chat Template 设计说明

### 核心原则

SFT 和推理时最重要的一条原则是：**训练时怎么包装，推理时就尽量怎么包装**。

LLM 本质上仍然是在做 next-token prediction。chat template 的作用不是让模型机制变成“真正问答系统”，而是把上下文组织成模型在 SFT 中见过的模式：

```text
<user>
用户问题
</user>
<assistant>
模型从这里开始续写
```

对当前 Qwen base model，优先使用 tokenizer 自带的 `apply_chat_template`。不要自己随便发明新的分隔符，除非你准备在 SFT 阶段也用同一套分隔符重新训练。

### 为什么不要直接裸输入问题

裸输入：

```text
There are 7 bags. Each bag has 6 candies. How many candies are there?
```

对 base model 来说，这只是普通文本前缀。它可能回答，也可能继续写另一个题目、补全文档、续写网页内容或重复输入。

chat template 输入：

```text
<|im_start|>user
There are 7 bags...<|im_end|>
<|im_start|>assistant
```

这会明确告诉模型：现在轮到 assistant 继续生成。

### 当前 GSM8K SFT 的推荐模板

当前数据集的 user 内容不是只有题目，还额外追加了格式要求：

```text
Let's think step by step and output the final answer after "####".
```

因此评估 GSM8K SFT 效果时，建议至少对比两种数学 prompt：

```text
A. 裸题目
B. 题目 + 与训练一致的格式指令
```

如果 B 的 `####` 格式遵循率明显高于 A，这是正常现象，不代表训练失败。它说明模型确实依赖训练时的条件分布。

### 设计 chat template 时的注意事项

1. 优先使用模型 tokenizer 自带的模板。
   对 Qwen 系列，使用 `tokenizer.apply_chat_template(...)`，不要手写 `<|im_start|>` 细节，除非你明确知道 tokenizer 的特殊 token 规则。

2. 推理时要加 generation prompt。
   也就是让序列停在 assistant 起点后：`add_generation_prompt=True`。否则模型可能不知道应该从哪里开始回答。

3. 训练时不要把 user token 算进 loss。
   SFT 的目标是让模型在看到 user 后生成 assistant，不是让模型背诵 user。训练代码里要把 prompt 部分 label 设为 `-100`。

4. 不要把 ground truth 泄漏进推理 prompt。
   评估 base/SFT 时，prompt 只能包含题目和格式要求，不能包含 assistant 标准答案。

5. system prompt 要保守。
   如果训练数据没有 system prompt，第一轮实验可以不加 system prompt，减少分布变化。等流程稳定后再测试加入系统提示是否有帮助。

6. Qwen3 的 thinking 模式要固定。
   当前实验建议 `enable_thinking=False`，让模板中显式使用非 thinking 生成格式，减少小模型输出过长和格式漂移。

7. 模板、训练数据、评估脚本必须一起记录。
   否则你看到的提升可能只是 prompt/template 改了，而不是 SFT 真正改进了模型。


In [ ]:
# 模块 6.2: Chat template helper
#
# 这一格只定义模板相关函数，不加载模型。
# 目标：保证 base 评估、SFT 训练、SFT 后评估使用同一套 prompt 构造逻辑。

FORMAT_INSTRUCTION = 'Let\'s think step by step and output the final answer after "####".'

def normalize_messages(messages):
    """把 parquet 读出来的 numpy.ndarray / list 统一成普通 Python list[dict]。

    pandas 读取 parquet 后，messages 可能是 numpy.ndarray。
    训练和模板函数更适合处理普通 list，所以这里统一转换。
    """
    if hasattr(messages, "tolist"):
        messages = messages.tolist()
    return [dict(m) for m in messages]

def build_gsm8k_user_content(question, include_format_instruction=True):
    """构造 GSM8K user 内容。

    include_format_instruction=True 时，和训练数据保持一致：题目后追加 #### 格式指令。
    include_format_instruction=False 时，用来观察模型在裸数学题上的泛化表现。
    """
    question = str(question).strip()
    if include_format_instruction:
        return question + " " + FORMAT_INSTRUCTION
    return question

def build_user_messages(question, include_format_instruction=True):
    """构造推理用 messages，只包含 user，不包含 assistant 标准答案。"""
    return [
        {
            "role": "user",
            "content": build_gsm8k_user_content(question, include_format_instruction),
        }
    ]

def apply_chat_template_text(tokenizer, messages, add_generation_prompt):
    """把 messages 渲染成模型真正看到的文本。

    Qwen3 tokenizer 支持 enable_thinking 参数。当前实验固定 enable_thinking=False，
    避免小模型在 SFT 前后混入过长 thinking 内容。
    如果某个 tokenizer 不接受 enable_thinking，则自动回退到普通 apply_chat_template。
    """
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

def render_generation_prompt(tokenizer, question, include_format_instruction=True):
    """渲染推理 prompt。

    add_generation_prompt=True 会让文本停在 assistant 起点后面，
    这正是 generate() 应该开始续写的位置。
    """
    messages = build_user_messages(question, include_format_instruction)
    return apply_chat_template_text(tokenizer, messages, add_generation_prompt=True)


## 模块 6.3 构造 held-out 小样本评估集

`train_500.parquet` 是训练小样本，不应该同时拿来判断 SFT 是否真的提升。这里从 `test.parquet` 里抽一个很小的 `eval_20.parquet`，用于：

```text
base 模型 SFT 前评估
SFT adapter 训练后评估
base vs sft 横向对比
```

第一轮只取 20 条，是为了快速观察格式、输出长度和 exact match 计算链路是否正常。后面可以扩大到 100/200 条。

In [ ]:
# 从 test.parquet 抽 held-out eval set。不要从 train_500 里抽评估集。
EVAL_N = 20
eval_file = GSM8K_SFT_DIR / f"eval_{EVAL_N}.parquet"

df_test = pd.read_parquet(test_file)
df_eval = df_test.head(EVAL_N).copy()
df_eval.to_parquet(eval_file)

print("saved:", eval_file)
print("rows:", len(df_eval))
print("first eval messages:")
print(df_eval.iloc[0]["messages"])


## 模块 6.4 SFT 前：base model 小样本评估

这一节用同一个 held-out `eval_20.parquet` 测 base model。目的不是得到严肃 benchmark，而是建立 SFT 前 baseline：

```text
base 的 exact match 是多少
base 是否输出 ####
base 平均输出多长
base 在裸题目和格式指令题目上的差异
```

注意：base model 本质上可能只是续写，所以如果它不稳定回答，这是预期现象。这个 baseline 用来和后面的 SFT adapter 对比。

In [ ]:
# 模块 6.5: base model 评估代码
#
# 这段会真正加载 base model 并生成答案。RTX 4070 上建议先保持 EVAL_N=20。
# 如果显存不足：
# - 降低 max_new_tokens，例如 128。
# - 评估后及时 del model 并 torch.cuda.empty_cache()。

import re
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def extract_final_answer(text):
    """从模型输出或标准答案中抽取最终数字答案。

    优先匹配 GSM8K 标准格式中的 ####。
    如果没有 ####，再退化为取最后一个数字；这样可以观察“答案可能对但格式不对”的情况。
    """
    text = str(text)
    match = re.search(r"####\s*(-?[0-9][0-9,]*(?:\.\d+)?)", text)
    if match:
        return match.group(1).replace(",", "")
    nums = re.findall(r"-?[0-9][0-9,]*(?:\.\d+)?", text)
    return nums[-1].replace(",", "") if nums else None

def generate_one(model, tokenizer, question, include_format_instruction=True, max_new_tokens=192):
    """对单条 GSM8K 问题生成回答。

    关键点：这里不是裸输入 question，而是先走 render_generation_prompt。
    这样 base 和 SFT 都在同一套 chat template 下比较。
    """
    prompt_text = render_generation_prompt(tokenizer, question, include_format_instruction)
    inputs = tokenizer(prompt_text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def evaluate_gsm8k_sample(model, tokenizer, eval_parquet, include_format_instruction=True, max_items=None, tag="base"):
    """在小样本 GSM8K eval parquet 上评估。

    输出两个东西：
    - summary: exact match、format rate、avg length。
    - rows: 每条样本的 question、gold、prediction，方便人工观察。
    """
    df = pd.read_parquet(eval_parquet)
    if max_items is not None:
        df = df.head(max_items)

    rows = []
    started = time.time()
    for idx, row in df.iterrows():
        messages = normalize_messages(row["messages"])
        user_content = messages[0]["content"]
        assistant_gold = messages[1]["content"]

        # eval parquet 的 user 已经带了格式指令。为了同时支持“裸题目”评估，
        # 这里把训练时追加的 FORMAT_INSTRUCTION 去掉，再由 generate_one 决定是否重新追加。
        question = user_content.replace(" " + FORMAT_INSTRUCTION, "").strip()

        prediction = generate_one(
            model,
            tokenizer,
            question,
            include_format_instruction=include_format_instruction,
        )

        gold_answer = extract_final_answer(assistant_gold)
        pred_answer = extract_final_answer(prediction)
        rows.append(
            {
                "idx": int(idx),
                "tag": tag,
                "include_format_instruction": include_format_instruction,
                "question": question,
                "gold_answer": gold_answer,
                "pred_answer": pred_answer,
                "exact_match": pred_answer == gold_answer,
                "format_ok": "####" in prediction,
                "pred_chars": len(prediction),
                "prediction": prediction,
                "gold": assistant_gold,
            }
        )

        print(f"[{len(rows)}/{len(df)}] gold={gold_answer} pred={pred_answer} format={'####' in prediction}")

    result_df = pd.DataFrame(rows)
    summary = {
        "tag": tag,
        "n": len(result_df),
        "include_format_instruction": include_format_instruction,
        "exact_match": float(result_df["exact_match"].mean()) if len(result_df) else 0.0,
        "format_rate": float(result_df["format_ok"].mean()) if len(result_df) else 0.0,
        "avg_chars": float(result_df["pred_chars"].mean()) if len(result_df) else 0.0,
        "seconds": round(time.time() - started, 2),
    }
    return summary, result_df

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
base_model.eval()

print("Rendered prompt preview:")
print(render_generation_prompt(tokenizer, "There are 7 bags. Each bag has 6 candies. How many candies are there?", True))

base_summary, base_eval_rows = evaluate_gsm8k_sample(
    base_model,
    tokenizer,
    eval_file,
    include_format_instruction=True,
    max_items=EVAL_N,
    tag="base_with_format_instruction",
)

print("summary:", base_summary)
base_eval_rows.head(3)


In [ ]:
# 可选：保存 base 评估结果，方便 SFT 后横向对比。
eval_output_dir = BASE_EVAL_OUTPUT_DIR
eval_output_dir.mkdir(parents=True, exist_ok=True)
base_eval_path = eval_output_dir / f"base_eval_{EVAL_N}.jsonl"
base_eval_rows.to_json(base_eval_path, orient="records", lines=True, force_ascii=False)
print("saved:", base_eval_path)

# 如果接下来要训练，为了释放显存，可以执行下面三行。
# del base_model
# import gc; gc.collect()
# torch.cuda.empty_cache()


## 模块 6.6 Attention Mask、Labels 与 Assistant-only Loss 笔记

这一节解释 SFT 训练里最容易混淆的三个东西：

```text
attention_mask: 控制哪些 token 能参与 attention 计算。
labels=-100: 控制哪些 token 不参与 loss。
assistant-only loss: 只优化 assistant 回答部分，不优化 user prompt 部分。
```

### 1. SFT 的训练目标是什么

一条 SFT 样本经过 chat template 后，可以简化成：

```text
[user prompt tokens] [assistant answer tokens]
```

例如：

```text
U1 U2 U3 <assistant> A1 A2 A3 <eos>
```

其中：

```text
U1 U2 U3 <assistant>  是条件上下文
A1 A2 A3 <eos>        是训练目标
```

SFT 要优化的是：

```text
P(assistant answer | user prompt)
```

而不是：

```text
P(user prompt + assistant answer)
```

所以 user 部分必须作为上下文输入模型，但不应该作为 loss 目标。

### 2. attention_mask 做什么

`attention_mask` 的主要作用是区分真实 token 和 padding token。

一个 batch 里样本长度可能不同：

```text
样本 1: U1 U2 A1 A2 <pad> <pad>
样本 2: U1 U2 U3 A1 A2 A3
```

对应的 `attention_mask` 是：

```text
样本 1: 1  1  1  1  0     0
样本 2: 1  1  1  1  1     1
```

含义：

```text
1 = 真实 token，可以参与 attention
0 = padding token，不能参与 attention
```

注意：user token 的 `attention_mask` 也应该是 1。因为 assistant 回答必须能看见 user 问题。如果把 user 的 attention_mask 设成 0，模型就看不到题目了。

### 3. attention 的数学形式

Transformer self-attention 的核心公式是：

```text
Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V
```

加入 mask 后，可以理解成：

```text
Attention(Q, K, V)
= softmax((QK^T / sqrt(d_k)) + mask) V
```

mask 会把不允许看的位置加上一个极小值，例如 `-inf`。softmax 后，这些位置的注意力权重就接近 0。

Causal LM 里通常同时有两种 attention 约束：

```text
causal mask : 不能看未来 token
padding mask: 不能看 pad token
```

所以当前位置 `A2` 可以看：

```text
U1 U2 U3 <assistant> A1 A2
```

但不能看未来的：

```text
A3 <eos>
```

也不能看 padding。

### 4. 为什么 attention_mask 不能用来只训练 assistant

这是一个常见误区。`attention_mask` 决定的是“看不看”，不是“算不算 loss”。

错误做法：

```text
user attention_mask = 0
assistant attention_mask = 1
```

这会让 assistant 看不到 user prompt，模型等于在不看题目的情况下硬背回答风格。

正确做法：

```text
user attention_mask      = 1
assistant attention_mask = 1
pad attention_mask       = 0

user labels      = -100
assistant labels = token id
pad labels       = -100
```

一句话：

```text
attention_mask 决定“看不看”
labels/loss_mask 决定“算不算”
```

### 5. labels=-100 为什么能忽略 user loss

Hugging Face / PyTorch 里常用 `-100` 作为 ignore index。它没有神秘数学含义，只是工程约定：正常 token id 通常是 `0 ~ vocab_size-1`，而 `-100` 不可能是合法 token id。

PyTorch 的 `CrossEntropyLoss` 默认：

```python
ignore_index = -100
```

也就是：

```text
if label[i] == -100:
    skip this position
else:
    compute cross_entropy(logits[i], label[i])
```

因此 SFT 里可以构造：

```text
tokens:
U1    U2    U3    <assistant>    A1    A2    A3

labels:
-100  -100  -100  -100           A1    A2    A3
```

最终只计算 assistant 部分的 cross entropy。

### 6. Loss 是逐 token 计算的

Causal LM 在每个位置都预测下一个 token。简化理解：

```text
看到 U1                 -> 预测 U2
看到 U1 U2              -> 预测 U3
看到 U1 U2 U3           -> 预测 <assistant>
看到 ... <assistant>    -> 预测 A1
看到 ... A1             -> 预测 A2
看到 ... A2             -> 预测 A3
```

但训练时 user 部分 label 是 `-100`，所以这些位置不产生 loss。assistant 部分保留真实 token id，所以会产生 loss。

概念上可以写成：

```text
loss = CE(A1) + CE(A2) + CE(A3) + CE(<eos>)
```

实际训练一般会除以 assistant token 数量，得到平均 loss。

### 7. Shift 细节

Causal LM 预测的是下一个 token，所以底层会做 shift：

```text
logits at position t -> predict label at position t+1
```

例如第一个 assistant token `A1` 的 loss，来自 `<assistant>` 位置的 logits：

```text
context: U1 U2 U3 <assistant>
target : A1
```

这正是我们想要的训练行为：在看到 user prompt 和 assistant 起点后，学会生成 assistant 的第一个回答 token。

### 8. 对模型参数更新的影响

因为 user 部分不计算 loss，所以参数更新的梯度直接来自 assistant token 的预测误差。

但这不表示 user 不重要。更准确地说：

```text
训练信号来自 assistant 部分；
assistant 部分的预测以 user 部分作为条件上下文。
```

所以 SFT 学到的是条件映射：

```text
看到这种 user 输入
-> 生成这种 assistant 输出
```

对 GSM8K 来说就是：

```text
看到数学应用题 + #### 格式指令
-> 生成逐步解题过程 + #### 最终答案
```

### 9. 和本 notebook 训练代码的对应关系

模块 7.1 的训练代码里：

```python
full_text = user + assistant answer
prompt_text = user + assistant generation prompt

labels = input_ids.copy()
labels[:prompt_len] = [-100] * prompt_len
```

含义就是：

```text
prompt_text 范围内的 token 不算 loss
prompt_text 之后的 assistant answer token 算 loss
```

collator 里又做了 padding：

```text
attention_mask padding 为 0
labels padding 为 -100
```

对应关系：

```text
padding 不参与 attention，也不参与 loss
user prompt 参与 attention，但不参与 loss
assistant answer 参与 attention，也参与 loss
```


## 模块 7. LoRA SFT 冒烟训练

这一节是真正的第一轮 SFT 训练模块。当前环境已经有 `peft` 和 `accelerate`，但没有 `trl` / `ray`。因此第一轮不走 verl SFT trainer，也不依赖 TRL，而是使用：

```text
Transformers Trainer + PEFT LoRA + 自定义 assistant-only labels
```

第一轮只用 `train_500.parquet`，目标是验证训练链路，不追求最终效果。

### 训练验收标准

```text
1. 能加载 base model 和 tokenizer
2. 能把 messages 渲染成 chat template
3. labels 中 prompt/user 部分是 -100，只训练 assistant 部分
4. LoRA trainable params 数量合理
5. loss 能正常打印
6. 能保存 adapter
7. SFT 后在同一 eval_20 上重新评估
```

如果 OOM，优先调整：

```text
max_length: 512 -> 384
gradient_accumulation_steps: 16 -> 32
lora_r: 16 -> 8
max_new_tokens: 192 -> 128
```


In [ ]:
# 模块 7.1: LoRA SFT 训练代码
#
# 这段会真正开始训练。建议确认 base 评估结果已经保存后再运行。
# 如果你刚刚还保留了 base_model，占用显存，请先运行上一格里的 del/empty_cache。

import gc
from dataclasses import dataclass
from torch.utils.data import Dataset as TorchDataset
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model

TRAIN_MAX_LENGTH = 512
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TRAIN_OUTPUT_DIR = OUTPUT_DIR / "qwen3_0d6b_gsm8k_lora_500"

class MessageSFTDataset(TorchDataset):
    """把 messages parquet 转成 CausalLM SFT 训练样本。

    每个样本包含：
    - input_ids: 完整 chat 文本，也就是 user + assistant。
    - attention_mask: 有效 token 位置。
    - labels: 与 input_ids 同长，但 user/prompt 部分设为 -100，只训练 assistant。
    """

    def __init__(self, parquet_path, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []

        df = pd.read_parquet(parquet_path)
        for row_id, row in df.iterrows():
            messages = normalize_messages(row["messages"])
            item = self._tokenize_one(messages)
            if item is not None:
                self.samples.append(item)

        print("loaded train samples:", len(self.samples), "from", parquet_path)

    def _tokenize_one(self, messages):
        # full_text 包含 user 和 assistant 标准答案，用作完整 input_ids。
        full_text = apply_chat_template_text(
            self.tokenizer,
            messages,
            add_generation_prompt=False,
        )

        # prompt_text 只包含 user，并停在 assistant 起点后。
        # 它的 token 长度就是需要 mask 掉、不参与 loss 的前缀长度。
        prompt_text = apply_chat_template_text(
            self.tokenizer,
            [messages[0]],
            add_generation_prompt=True,
        )

        full_tokens = self.tokenizer(
            full_text,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_length,
        )
        prompt_tokens = self.tokenizer(
            prompt_text,
            add_special_tokens=False,
            truncation=False,
        )

        input_ids = full_tokens["input_ids"]
        attention_mask = full_tokens["attention_mask"]
        labels = list(input_ids)

        # 只训练 assistant 部分：prompt/user 位置 label=-100，Trainer 会忽略这些 token 的 loss。
        prompt_len = min(len(prompt_tokens["input_ids"]), len(labels))
        labels[:prompt_len] = [-100] * prompt_len

        # 如果 max_length 太短，可能把 assistant 全截掉。这样的样本没有训练信号，直接跳过。
        if all(x == -100 for x in labels):
            return None

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

@dataclass
class CausalLMCollator:
    """动态 padding collator。

    input_ids 用 tokenizer.pad_token_id padding；
    attention_mask padding 为 0；
    labels padding 为 -100，避免 padding token 参与 loss。
    """
    tokenizer: object

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        pad_id = self.tokenizer.pad_token_id

        for f in features:
            seq_len = len(f["input_ids"])
            pad_len = max_len - seq_len
            batch["input_ids"].append(torch.cat([f["input_ids"], torch.full((pad_len,), pad_id, dtype=torch.long)]))
            batch["attention_mask"].append(torch.cat([f["attention_mask"], torch.zeros((pad_len,), dtype=torch.long)]))
            batch["labels"].append(torch.cat([f["labels"], torch.full((pad_len,), -100, dtype=torch.long)]))

        return {k: torch.stack(v) for k, v in batch.items()}

# 重新加载 tokenizer，确保训练代码可以独立运行。
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset = MessageSFTDataset(small_train_file, tokenizer, max_length=TRAIN_MAX_LENGTH)
collator = CausalLMCollator(tokenizer)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model.config.use_cache = False

# Qwen 系列常见线性层名称。LoRA 只训练这些低秩 adapter，不更新全部 base 参数。
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=str(TRAIN_OUTPUT_DIR),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    learning_rate=1e-4,
    logging_steps=5,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    optim="adamw_torch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator,
)

# 真正开始训练。第一轮 500 条样本通常只用于冒烟测试。
trainer.train()

# 保存 LoRA adapter 和 tokenizer。这里保存的是 adapter，不是合并后的完整模型。
trainer.save_model(str(TRAIN_OUTPUT_DIR))
tokenizer.save_pretrained(str(TRAIN_OUTPUT_DIR))
print("saved LoRA adapter to:", TRAIN_OUTPUT_DIR)


## 模块 7.2 SFT 后评估：加载 LoRA adapter 对比 base

训练完成后，在同一个 `eval_20.parquet` 上重新评估 SFT adapter。重点不是只看 exact match，还要看：

```text
format_rate 是否上升
avg_chars 是否异常变长
是否更稳定输出 ####
普通问答是否被污染成数学格式
```

如果 base 的 `format_rate` 很低，SFT 后明显升高，这说明 SFT 至少学到了格式和任务分布。正确率是否提升，需要更大的 held-out eval 才能判断。

In [ ]:
# 模块 7.3: SFT adapter 小样本评估
#
# 如果刚训练完，model 变量已经是带 LoRA 的模型，可以直接用它评估。
# 如果你重启了 notebook，则取消下面 PeftModel 加载逻辑的注释重新加载 adapter。

from peft import PeftModel

# 重启 notebook 后使用：
# tokenizer = AutoTokenizer.from_pretrained(TRAIN_OUTPUT_DIR, trust_remote_code=True)
# if tokenizer.pad_token_id is None:
#     tokenizer.pad_token = tokenizer.eos_token
# base_for_sft = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL_DIR,
#     torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
#     device_map="auto" if torch.cuda.is_available() else None,
#     trust_remote_code=True,
# )
# model = PeftModel.from_pretrained(base_for_sft, TRAIN_OUTPUT_DIR)

model.eval()
sft_summary, sft_eval_rows = evaluate_gsm8k_sample(
    model,
    tokenizer,
    eval_file,
    include_format_instruction=True,
    max_items=EVAL_N,
    tag="sft_lora_500_with_format_instruction",
)

print("base summary:", base_summary if "base_summary" in globals() else "base_summary not in memory; load base_eval jsonl if needed")
print("sft summary:", sft_summary)

sft_eval_path = SFT_EVAL_OUTPUT_DIR / f"sft_lora_500_eval_{EVAL_N}.jsonl"
sft_eval_rows.to_json(sft_eval_path, orient="records", lines=True, force_ascii=False)
print("saved:", sft_eval_path)
sft_eval_rows.head(3)


## 模块 7.4 SFT 输出复读诊断

这一节不重新生成，只读取已经保存的 `sft_lora_500_eval_20.jsonl`，统计每条输出里 `####` 和 `The answer is` 的重复次数。目标是区分两个现象：格式遵循率上升，以及生成是否在最终答案后继续复读。

In [ ]:
# 模块 7.4: 读取已保存的 SFT 评估结果，统计复读现象

import re
import pandas as pd

sft_eval_path = SFT_EVAL_OUTPUT_DIR / f"sft_lora_500_eval_{EVAL_N}.jsonl"
sft_eval_rows_saved = pd.read_json(sft_eval_path, lines=True)

def count_text_pattern(text, pattern):
    """统计一段文本中某个正则模式出现了多少次。"""
    return len(re.findall(pattern, str(text)))

diagnosis_rows = []
for _, row in sft_eval_rows_saved.iterrows():
    prediction = str(row["prediction"])
    hash_count = count_text_pattern(prediction, r"####")
    answer_is_count = count_text_pattern(prediction, r"The answer is")
    diagnosis_rows.append(
        {
            "idx": int(row["idx"]),
            "gold_answer": row["gold_answer"],
            "pred_answer": row["pred_answer"],
            "exact_match": bool(row["exact_match"]),
            "format_ok": bool(row["format_ok"]),
            "pred_chars": int(row["pred_chars"]),
            "hash_count": hash_count,
            "answer_is_count": answer_is_count,
            "repeat_like": hash_count > 1 or answer_is_count > 1,
        }
    )

diagnosis_df = pd.DataFrame(diagnosis_rows)
diagnosis_summary = {
    "n": len(diagnosis_df),
    "exact_match": float(diagnosis_df["exact_match"].mean()),
    "format_rate": float(diagnosis_df["format_ok"].mean()),
    "repeat_like_rate": float(diagnosis_df["repeat_like"].mean()),
    "avg_chars": float(diagnosis_df["pred_chars"].mean()),
    "max_hash_count": int(diagnosis_df["hash_count"].max()),
}

print("diagnosis summary:", diagnosis_summary)
diagnosis_df.sort_values(
    ["repeat_like", "hash_count", "pred_chars"],
    ascending=[False, False, False],
).head(20)


## 模块 7.4.5 加载指定的 LoRA SFT 模型

这个模块独立于训练流程，专门用于从磁盘加载已训练好的 LoRA adapter。每次要评估不同 adapter 时，只需修改 `ADAPTER_DIR` 路径，然后顺序运行本模块 + 7.5（或 7.3）。

**设计原则**：
- tokenizer 从 adapter 目录加载（包含训练时保存的 `chat_template.jinja`），保证推理模板与训练一致
- base model 从 `BASE_MODEL_DIR` 加载，再通过 `PeftModel.from_pretrained` 挂载 LoRA adapter
- 加载前自动释放旧模型显存，避免 OOM

**当前可用的 adapter 目录**：

| 训练数据 | 路径 |
|----------|------|
| 500 条冒烟 | `models/sft/qwen3_0d6b_gsm8k_lora_500` |
| 完整 7470 条 | `models/sft/qwen3_0d6b_gsm8k_lora_full_20260611_000923` |

In [ ]:
# 模块 7.4.5: 加载指定的 LoRA SFT adapter
#
# 使用说明：
# 1. 修改下面 ADAPTER_DIR 为你要评估的 adapter 路径。
# 2. 运行本单元格。
# 3. 直接运行模块 7.5（或 7.3），会用刚加载的 model/tokenizer 做评估。
#
# 可选 adapter：
#   OUTPUT_DIR / "qwen3_0d6b_gsm8k_lora_500"                      # 500 条冒烟
#   OUTPUT_DIR / "qwen3_0d6b_gsm8k_lora_full_20260611_000923"      # 完整 7470 条

import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ============================================================
# 在这里指定要加载的 adapter 目录
# ============================================================
ADAPTER_DIR = OUTPUT_DIR / "qwen3_0d6b_gsm8k_lora_full_20260611_000923"

# 确保 eval_file / EVAL_N 已定义，避免 7.5 报 NameError。
EVAL_N = 20
eval_file = GSM8K_SFT_DIR / f"eval_{EVAL_N}.parquet"

# 1. 释放当前内存中的旧模型，避免显存叠加导致 OOM。
objects_to_unload = [
    "model",
    "base_model",
    "base_for_sft",
    "trainer",
    "full_trainer",
    "train_dataset",
    "full_train_dataset",
]
for name in objects_to_unload:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("cuda allocated after unload (GB):", round(torch.cuda.memory_allocated() / 1024**3, 3))

# 2. 从 adapter 目录加载 tokenizer，保证 chat_template 与训练时一致。
tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3. 加载 base model 再挂载 LoRA adapter。
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model.eval()

print("=" * 60)
print("adapter:", ADAPTER_DIR)
print("base model:", BASE_MODEL_DIR)
print("cuda allocated (GB):", round(torch.cuda.memory_allocated() / 1024**3, 3))
print("=" * 60)
print("已就绪，可以运行模块 7.5（或 7.3）做评估。")

## 模块 7.5 短输出完整评估与复读统计

这一节重新生成一遍 SFT 输出：把 `max_new_tokens` 降到 96，但完整保留 `prediction`。不要在第一个 `#### 数字` 后截断主输出，否则会掩盖最终答案之后是否仍然复读。

评估时把两个问题分开看：

```text
答案正确性：只从第一个合法 #### 数字抽取 first_hash_answer，再和 gold_answer 比较。
格式和复读：必须基于完整 prediction 统计，不能基于截断文本统计。
```

重点看这些指标：

```text
format_rate              是否至少出现一个合法 #### 数字
single_final_answer_rate 是否只有一个 #### 数字且只有一个 ####
repeat_like_rate         是否出现多个 ####、重复 The answer is 或重复行
avg_hash_count           平均每条输出出现多少个 ####
max_hash_count           最严重样本里 #### 出现多少次
avg_chars                完整输出平均长度
first_hash_exact_match   第一个 #### 数字是否等于标准答案
```

判断标准：

```text
format_rate 高 + repeat_like_rate 低 + single_final_answer_rate 高
=> 主要是原来 max_new_tokens 太长，短输出和停止 token 基本能控制住复读。

format_rate 高 + repeat_like_rate 仍高 + single_final_answer_rate 低
=> 模型学到了表面格式，但停止边界没学稳，建议扩大 SFT 数据到 1k-5k 再训。

exact_match 低 + repeat_like_rate 高
=> 当前 500 条更像学到模板，不足以稳定学会 GSM8K 解题分布。
```

这一节会把完整结果保存为 `sft_lora_500_eval_20_max96_full.jsonl`，用于后续和 `sft_lora_500_eval_20.jsonl` 横向比较。

In [ ]:
# 模块 7.5: 使用更短生成长度，完整保存输出并统计复读
#
# 运行前提：当前 kernel 里已经有 model、tokenizer、eval_file、EVAL_N。
# 如果重启过 notebook，先运行 7.4.5 加载 LoRA adapter。

import re
import time
import torch
import pandas as pd

# ============================================================
# 在这里设置生成长度上限（每次运行前手动修改）
#  96 = 短输出（快速诊断复读是否被压制）
# 192 = 长输出（给足推理空间，format_rate 更反映真实格式学习效果）
# ============================================================
SHORT_MAX_NEW_TOKENS = 192

FINAL_ANSWER_PATTERN = r"####\s*(-?[0-9][0-9,]*(?:\.\d+)?)"

def count_text_pattern(text, pattern):
    """统计一段文本中某个正则模式出现了多少次。"""
    return len(re.findall(pattern, str(text)))

def build_eos_token_ids(tokenizer):
    """收集常见停止 token，帮助模型在 assistant 轮次结束处停下。"""
    token_ids = []
    if tokenizer.eos_token_id is not None:
        token_ids.append(tokenizer.eos_token_id)

    for token in ["<|im_end|>", "<|endoftext|>"]:
        token_id = tokenizer.convert_tokens_to_ids(token)
        if isinstance(token_id, int) and token_id >= 0:
            token_ids.append(token_id)

    token_ids = sorted(set(token_ids))
    if not token_ids:
        return None
    return token_ids[0] if len(token_ids) == 1 else token_ids

def extract_first_hash_answer(text):
    """只从第一个 #### 数字中抽取答案；不截断原始输出。"""
    match = re.search(FINAL_ANSWER_PATTERN, str(text))
    if not match:
        return None
    return match.group(1).replace(",", "")

def summarize_repetition(text):
    """基于完整输出统计复读信号。"""
    text = str(text)
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    repeated_line_count = len(lines) - len(set(lines))
    hash_count = count_text_pattern(text, r"####")
    final_answer_count = count_text_pattern(text, FINAL_ANSWER_PATTERN)
    answer_is_count = count_text_pattern(text, r"The answer is")
    repeat_like = hash_count > 1 or answer_is_count > 1 or repeated_line_count > 0
    return {
        "hash_count": hash_count,
        "final_answer_count": final_answer_count,
        "answer_is_count": answer_is_count,
        "repeated_line_count": repeated_line_count,
        "repeat_like": repeat_like,
    }

def generate_one_controlled(model, tokenizer, question, include_format_instruction=True, max_new_tokens=96):
    """使用较短生成长度和显式停止 token 生成单条回答。"""
    prompt_text = render_generation_prompt(tokenizer, question, include_format_instruction)
    inputs = tokenizer(prompt_text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    eos_token_id = build_eos_token_ids(tokenizer)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=eos_token_id,
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def evaluate_gsm8k_sample_controlled(
    model,
    tokenizer,
    eval_parquet,
    include_format_instruction=True,
    max_items=None,
    max_new_tokens=96,
    tag="sft_short_full",
):
    """重新评估 GSM8K，完整保存输出，并基于完整输出统计格式和复读。"""
    df = pd.read_parquet(eval_parquet)
    if max_items is not None:
        df = df.head(max_items)

    rows = []
    started = time.time()
    for idx, row in df.iterrows():
        messages = normalize_messages(row["messages"])
        user_content = messages[0]["content"]
        assistant_gold = messages[1]["content"]
        question = user_content.replace(" " + FORMAT_INSTRUCTION, "").strip()

        raw_prediction = generate_one_controlled(
            model,
            tokenizer,
            question,
            include_format_instruction=include_format_instruction,
            max_new_tokens=max_new_tokens,
        )
        prediction = raw_prediction

        gold_answer = extract_final_answer(assistant_gold)
        first_hash_answer = extract_first_hash_answer(prediction)
        pred_answer = first_hash_answer if first_hash_answer is not None else extract_final_answer(prediction)
        repetition = summarize_repetition(prediction)
        format_ok = first_hash_answer is not None
        single_final_answer_ok = repetition["final_answer_count"] == 1 and repetition["hash_count"] == 1

        # 注意：prediction 保存完整输出，不做截断；first_hash_answer 只用于抽取第一个 #### 数字。
        # 这样既能判断答案是否正确，也能判断 max_new_tokens 降低后是否仍然复读。
        row_data = {
            "idx": int(idx),
            "tag": tag,
            "include_format_instruction": include_format_instruction,
            "max_new_tokens": max_new_tokens,
            "question": question,
            "gold_answer": gold_answer,
            "first_hash_answer": first_hash_answer,
            "pred_answer": pred_answer,
            "exact_match": pred_answer == gold_answer,
            "first_hash_exact_match": first_hash_answer == gold_answer,
            "format_ok": format_ok,
            "single_final_answer_ok": single_final_answer_ok,
            "hash_count": repetition["hash_count"],
            "final_answer_count": repetition["final_answer_count"],
            "answer_is_count": repetition["answer_is_count"],
            "repeated_line_count": repetition["repeated_line_count"],
            "repeat_like": repetition["repeat_like"],
            "pred_chars": len(prediction),
            "prediction": prediction,
            "gold": assistant_gold,
        }
        rows.append(row_data)

        print(
            f"[{len(rows)}/{len(df)}] gold={gold_answer} pred={pred_answer} "
            f"format={format_ok} repeat={repetition['repeat_like']} "
            f"hashes={repetition['hash_count']} finals={repetition['final_answer_count']} "
            f"chars={len(prediction)}"
        )

    result_df = pd.DataFrame(rows)
    summary = {
        "tag": tag,
        "n": len(result_df),
        "include_format_instruction": include_format_instruction,
        "max_new_tokens": max_new_tokens,
        "exact_match": float(result_df["exact_match"].mean()) if len(result_df) else 0.0,
        "first_hash_exact_match": float(result_df["first_hash_exact_match"].mean()) if len(result_df) else 0.0,
        "format_rate": float(result_df["format_ok"].mean()) if len(result_df) else 0.0,
        "single_final_answer_rate": float(result_df["single_final_answer_ok"].mean()) if len(result_df) else 0.0,
        "repeat_like_rate": float(result_df["repeat_like"].mean()) if len(result_df) else 0.0,
        "avg_hash_count": float(result_df["hash_count"].mean()) if len(result_df) else 0.0,
        "max_hash_count": int(result_df["hash_count"].max()) if len(result_df) else 0,
        "avg_chars": float(result_df["pred_chars"].mean()) if len(result_df) else 0.0,
        "seconds": round(time.time() - started, 2),
    }
    return summary, result_df

model.eval()
sft_short_full_summary, sft_short_full_rows = evaluate_gsm8k_sample_controlled(
    model,
    tokenizer,
    eval_file,
    include_format_instruction=True,
    max_items=EVAL_N,
    max_new_tokens=SHORT_MAX_NEW_TOKENS,
    tag=f"sft_lora_full_max{SHORT_MAX_NEW_TOKENS}_full",
)

sft_short_full_path = eval_output_dir / f"sft_lora_full_eval_{EVAL_N}_max{SHORT_MAX_NEW_TOKENS}_full.jsonl"
sft_short_full_rows.to_json(sft_short_full_path, orient="records", lines=True, force_ascii=False)
print("short/full summary:", sft_short_full_summary)
print("saved:", sft_short_full_path)

# 汇总指标字段说明：解释上面 short/full summary 字典中的关键指标。
summary_metric_descriptions = pd.DataFrame(
    [
        {"字段": "exact_match", "含义": "按 pred_answer 与 gold_answer 比较得到的整体正确率；pred_answer 优先来自第一个 #### 数字。"},
        {"字段": "first_hash_exact_match", "含义": "只看第一个合法 #### 数字是否等于标准答案；更严格地绑定到格式输出。"},
        {"字段": "format_rate", "含义": "有多少样本至少输出了一个合法的 #### 数字。"},
        {"字段": "single_final_answer_rate", "含义": "有多少样本只输出了一个 ####，且只包含一个合法最终答案。越高说明停止边界越稳定。"},
        {"字段": "repeat_like_rate", "含义": "有多少样本疑似复读；触发条件包括多个 ####、多次 The answer is 或重复行。"},
        {"字段": "avg_hash_count", "含义": "平均每条输出中 #### 出现次数。大于 1 通常说明最终答案后仍在继续生成。"},
        {"字段": "max_hash_count", "含义": "单条样本中 #### 出现的最大次数，用来定位最严重的复读样本。"},
        {"字段": "avg_chars", "含义": "完整 prediction 的平均字符数，不是截断后的长度。"},
        {"字段": "seconds", "含义": "本轮 20 条样本评估耗时。"},
    ]
)
display(summary_metric_descriptions)

# 每条样本结果表字段说明：解释下面 summary_columns 表格中的每个表头。
sample_column_descriptions = pd.DataFrame(
    [
        {"字段": "idx", "含义": "样本在 eval parquet 中的行号。"},
        {"字段": "gold_answer", "含义": "标准答案中从 #### 后抽取出的最终数字。"},
        {"字段": "pred_answer", "含义": "模型预测答案；优先取第一个合法 #### 数字，没有时退化为输出中的最后一个数字。"},
        {"字段": "exact_match", "含义": "pred_answer 是否等于 gold_answer。"},
        {"字段": "format_ok", "含义": "完整 prediction 中是否至少出现一个合法的 #### 数字。"},
        {"字段": "single_final_answer_ok", "含义": "完整 prediction 中是否只有一个 ####，且只包含一个合法最终答案。"},
        {"字段": "repeat_like", "含义": "是否疑似复读；多个 ####、多次 The answer is 或重复行都会标记为 True。"},
        {"字段": "hash_count", "含义": "完整 prediction 中 #### 出现的次数。"},
        {"字段": "final_answer_count", "含义": "完整 prediction 中合法 #### 数字模式出现的次数。"},
        {"字段": "answer_is_count", "含义": "完整 prediction 中 The answer is 出现的次数。"},
        {"字段": "repeated_line_count", "含义": "完整 prediction 中重复行的数量；大于 0 说明有逐行复读迹象。"},
        {"字段": "pred_chars", "含义": "完整 prediction 的字符数。"},
        {"字段": "prediction", "含义": "模型完整输出文本；用于人工确认是否复读、乱码或输出过长。"},
    ]
)
display(sample_column_descriptions)

summary_columns = [
    "idx",
    "gold_answer",
    "pred_answer",
    "exact_match",
    "format_ok",
    "single_final_answer_ok",
    "repeat_like",
    "hash_count",
    "final_answer_count",
    "answer_is_count",
    "repeated_line_count",
    "pred_chars",
]
display(sft_short_full_rows[summary_columns].head(20))

# 完整文本视图：用于人工确认 max_new_tokens 降低后是否仍然复读。
with pd.option_context("display.max_colwidth", None):
    display(sft_short_full_rows[["idx", "gold_answer", "pred_answer", "format_ok", "repeat_like", "prediction"]].head(20))


## 模块 7.6 使用完整 train.parquet 重新训练 LoRA SFT

这一节使用完整 `train.parquet` 重新开始一轮 LoRA SFT。运行前会先卸载当前 notebook 中已经加载的 `model`、`base_model`、`trainer` 等大对象，并清理 CUDA 显存，避免把上一轮 500 条样本训练或评估时加载的模型继续留在显存里。

注意：这里不会覆盖 `qwen3_0d6b_gsm8k_lora_500`，而是用时间戳创建新的输出目录，例如：

```text
models/sft/qwen3_0d6b_gsm8k_lora_full_YYYYMMDD_HHMMSS
```

训练完成后，可以直接重新运行模块 7.5，用当前内存中的 full SFT `model` 做 `eval_20` 评估，重点比较 `format_rate`、`single_final_answer_rate`、`repeat_like_rate` 和 `first_hash_exact_match`。

In [ ]:
# 模块 7.6: 使用完整 train.parquet 重新开始一轮 LoRA SFT
#
# 运行前提：前面已经执行过数据和训练辅助类定义，包括：
# - BASE_MODEL_DIR / GSM8K_SFT_DIR / OUTPUT_DIR
# - MessageSFTDataset
# - CausalLMCollator
# - normalize_messages / apply_chat_template_text

import gc
import time
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model

required_symbols = [
    "BASE_MODEL_DIR",
    "GSM8K_SFT_DIR",
    "OUTPUT_DIR",
    "MessageSFTDataset",
    "CausalLMCollator",
    "normalize_messages",
    "apply_chat_template_text",
]
missing_symbols = [name for name in required_symbols if name not in globals()]
if missing_symbols:
    raise RuntimeError(f"请先运行前面的基础定义单元，缺少这些变量或类: {missing_symbols}")

# 1. 先卸载当前已经加载的大对象，确保这一轮训练从 base model 重新开始。
#    这里只删除 notebook 变量，不删除磁盘上的 checkpoint 或评估结果。
objects_to_unload = [
    "model",
    "base_model",
    "base_for_sft",
    "trainer",
    "full_trainer",
    "train_dataset",
    "full_train_dataset",
    "sft_eval_rows",
    "sft_short_full_rows",
]
for name in objects_to_unload:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception as exc:
        print("skip cuda ipc_collect:", exc)
    print("cuda allocated GB after unload:", round(torch.cuda.memory_allocated() / 1024**3, 3))
    print("cuda reserved  GB after unload:", round(torch.cuda.memory_reserved() / 1024**3, 3))

# 2. 完整训练数据和本轮输出目录。
FULL_TRAIN_FILE = GSM8K_SFT_DIR / "train.parquet"
if not FULL_TRAIN_FILE.exists():
    raise FileNotFoundError(FULL_TRAIN_FILE)

FULL_RUN_NAME = f"qwen3_0d6b_gsm8k_lora_full_{time.strftime('%Y%m%d_%H%M%S')}"
FULL_TRAIN_OUTPUT_DIR = OUTPUT_DIR / FULL_RUN_NAME
FULL_TRAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

full_train_row_count = len(pd.read_parquet(FULL_TRAIN_FILE, columns=["messages"]))
print("full train file:", FULL_TRAIN_FILE)
print("full train rows:", full_train_row_count)
print("full output dir:", FULL_TRAIN_OUTPUT_DIR)

# 3. 完整数据集训练超参。相比 500 条冒烟训练，完整数据使用更保守的学习率。
FULL_TRAIN_MAX_LENGTH = 512
FULL_LORA_R = 16
FULL_LORA_ALPHA = 32
FULL_LORA_DROPOUT = 0.05
FULL_LEARNING_RATE = 5e-5
FULL_NUM_TRAIN_EPOCHS = 1
FULL_PER_DEVICE_BATCH_SIZE = 1
FULL_GRADIENT_ACCUMULATION_STEPS = 16

# 4. 从 base model 重新加载 tokenizer 和模型，不复用上一轮 LoRA model。
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

full_train_dataset = MessageSFTDataset(
    FULL_TRAIN_FILE,
    tokenizer,
    max_length=FULL_TRAIN_MAX_LENGTH,
)
full_collator = CausalLMCollator(tokenizer)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model.config.use_cache = False

# 5. 只训练 LoRA adapter，不更新完整 base model 权重。
full_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=FULL_LORA_R,
    lora_alpha=FULL_LORA_ALPHA,
    lora_dropout=FULL_LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, full_lora_config)
model.print_trainable_parameters()

full_training_args = TrainingArguments(
    output_dir=str(FULL_TRAIN_OUTPUT_DIR),
    per_device_train_batch_size=FULL_PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=FULL_GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=FULL_NUM_TRAIN_EPOCHS,
    learning_rate=FULL_LEARNING_RATE,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    optim="adamw_torch",
)

full_trainer = Trainer(
    model=model,
    args=full_training_args,
    train_dataset=full_train_dataset,
    data_collator=full_collator,
)

# 6. 正式开始完整数据训练。resume_from_checkpoint=False 表示不从旧 checkpoint 续训。
full_trainer.train(resume_from_checkpoint=False)

# 7. 保存完整数据训练得到的 LoRA adapter 和 tokenizer。
full_trainer.save_model(str(FULL_TRAIN_OUTPUT_DIR))
tokenizer.save_pretrained(str(FULL_TRAIN_OUTPUT_DIR))
trainer = full_trainer

print("saved full LoRA adapter to:", FULL_TRAIN_OUTPUT_DIR)
print("训练完成后，重新运行模块 7.5，可用当前内存中的 model 评估 full SFT 效果。")


## 模块 7.7 SFT 格式与复读问题排查复盘

### 7.7.1 当前现象

完整 SFT 后，模型比 500 条 SFT 更会做 GSM8K，但仍有明显复读和异常尾巴。按 `max_new_tokens` 对比，现象是：

```text
max_new_tokens 越长
-> 模型越有机会生成到 #### final_answer
-> exact_match / format_rate 往往更高
-> 但答案后继续生成的空间也更大
-> repeat_like_rate / avg_hash_count 也更高
```

这说明问题不能只靠调短 `max_new_tokens` 解决。调短会减少复读，但也会让很多样本还没输出 `####` 就被截断。SFT 的基本目标应该是：

```text
题目 -> 推理过程 -> #### final_answer -> <|im_end|>
```

而不是：

```text
题目 -> 推理过程 -> #### final_answer -> 继续复读/乱码/新题目
```

### 7.7.2 先固定评估口径

不要只看 `format_rate`，也不要只看 `exact_match`。每次 SFT 后至少记录：

```text
exact_match              第一个预测答案是否等于 gold_answer
format_rate              是否至少输出一个合法 #### 数字
single_final_answer_rate 是否只输出一个 #### 且只有一个合法最终答案
repeat_like_rate         是否出现多个 ####、重复 The answer is 或重复行
avg_hash_count           平均每条输出有多少个 ####
max_hash_count           最严重样本的 #### 次数
avg_chars                完整输出平均字符数
```

推荐同时跑几个生成长度：

```text
max_new_tokens = 96 / 128 / 160 / 192
```

判断方式：

```text
format_rate 高 + single_final_answer_rate 高 + repeat_like_rate 低
=> 模型比较稳定，基本学会了格式和停止边界。

format_rate 高 + repeat_like_rate 高
=> 学到了 #### 表面格式，但没有学稳答案后停止。

repeat_like_rate 低 + format_rate 低
=> 可能只是 max_new_tokens 太短，把输出提前截断了。
```

本轮完整 SFT 的观察是：`192` 正确率最高，但复读也最高；`160` 是临时折中点，但仍不算合格。

### 7.7.3 排查方向 A：训练数据是否干净

要先确认 `train.parquet` 里的 assistant gold 本身没有污染。检查项：

```text
1. messages 是否都是 user -> assistant
2. assistant 是否都有合法 #### 数字
3. assistant 是否有多个 #### 数字
4. 第一个 #### 数字后是否还有尾巴
5. 是否命中生成中出现过的异常片段，如 erotique / beurette / externalActionCode
```

执行脚本：

```powershell
D:\Anaconda\envs\test3\python.exe D:\learnAI\verl\yhy\scripts\sft\check_sft_data_and_labels.py
```

本轮检查结果：

```text
总样本数: 7473
role 不是 user->assistant: 0
缺少合法 #### 数字: 0
assistant 中出现多个合法 #### 数字: 0
第一个 #### 数字之后仍有尾巴: 0
命中已知异常片段: 0
```

结论：当前 `train.parquet` 基本干净，生成里的异常多语种尾巴不是直接从训练数据复制来的。

### 7.7.4 排查方向 B：max_length 是否截断 assistant 结尾

SFT 不只要训练 `#### answer`，还要训练 assistant 结束 token。若样本被 `max_length` 截断，模型看不到结尾的 `<|im_end|>`，就学不到在答案后停止。

检查逻辑：

```text
full_text   = chat_template(user + assistant, add_generation_prompt=False)
prompt_text = chat_template(user, add_generation_prompt=True)
full_ids    = tokenizer(full_text)
prompt_ids  = tokenizer(prompt_text)

labels = full_ids
labels[:len(prompt_ids)] = -100

如果 full_ids 被 max_length 截断，且 labels 区域没有 <|im_end|>，
说明这个样本没有教模型如何结束 assistant 回答。
```

本轮检查结果：

```text
max_length=512:
  被截断样本: 5 / 7473 = 0.07%
  label 区域没有结束 token: 5 / 7473 = 0.07%

max_length=768:
  被截断样本: 0
  label 区域没有结束 token: 0

max_length=1024:
  被截断样本: 0
  label 区域没有结束 token: 0
```

结论：`max_length=512` 有少量瑕疵，但只有 5 条，不足以解释大面积复读。后续正式训练可以用 `768`，彻底避免这类截断。

### 7.7.5 排查方向 C：label 尾部 decode 检查

这是确认 label mask 是否正确的关键检查。它不是看原始文本，而是看模型真正被训练预测的 token 尾部。

检查方法：

```text
1. 按训练代码构造 full_text 和 prompt_text。
2. 把 prompt_len 之前的 labels 设为 -100。
3. 只取 labels != -100 的 assistant 训练 token。
4. decode 最后几十个 token。
5. 看尾部是否是 #### 答案 <|im_end|>。
```

为什么有效：

```text
如果 label 尾部包含 <|im_end|>，说明 loss 确实在训练模型：答案后应该结束。
如果 label 尾部没有 <|im_end|>，或 <|im_end|> 被 mask 掉，模型就没有被教会停止。
```

本轮样本检查结果：

```text
idx=0:   #### 72<|im_end|>
idx=1:   #### 10<|im_end|>
idx=10:  #### 121<|im_end|>
```

这些普通样本正常。被 `512` 截断的 `idx=310` 在 `512` 下没有 `<|im_end|>`，但在 `768` 下正常：

```text
max_length=512: 截断在半截推理中，没有 <|im_end|>
max_length=768: #### 22000<|im_end|>
```

结论：训练 label mask 大体正确，结束 token 确实参与了绝大多数样本的 loss。

### 7.7.6 当前判断

根据目前证据：

```text
不是训练数据明显污染。
不是大面积 max_length 截断。
不是 label mask 完全错误。
```

更可能的原因是：

```text
1. 0.6B base 模型能力和停止控制较弱。
2. LoRA SFT 只做 next-token imitation，没有显式惩罚答案后的续写。
3. GSM8K 输出较长，模型在长生成时更容易进入重复循环。
4. 生成时没有强停止规则，max_new_tokens 越长越容易暴露这个问题。
```

### 7.7.7 下一步动作建议

优先级从高到低：

```text
1. 正式训练把 max_length 从 512 提到 768，消除少量结尾截断。
2. 评估继续保存完整 prediction，不要只看截断后的文本。
3. 推理侧临时使用 max_new_tokens=160 作为折中点。
4. 产品/展示侧可以在第一个合法 #### 数字后截断，但学习评估仍要看完整输出。
5. 如果继续训练，比较 768 max_length + full data 的 repeat_like_rate 是否下降。
6. 进入 RLHF/GRPO 前，可以把重复、多个 ####、答案后尾巴作为 rule penalty 或 hard negative。
```

阶段性验收标准可以先设为：

```text
format_rate >= 0.8
single_final_answer_rate 明显上升
repeat_like_rate <= 0.2
exact_match 不低于当前 full SFT max160/max192 水平
人工观察样本没有大面积乱码和无限复读
```

注意：SFT 只能学习“模仿答案格式和解题分布”，不能天然惩罚答案后的坏续写。后续 RM/RLHF 阶段可以把“答案后复读、多次 ####、格式正确但答案错”构造成 rejected/hard negative。

## 附录 A. verl SFT trainer 命令记录区

这里先记录命令，不自动执行。第一次建议先用 README 中更稳的 Transformers + TRL + LoRA 路线；等流程跑通后，再接 verl SFT trainer。

### A.1 verl SFT trainer 命令模板

```powershell
cd D:\learnAI\verl

D:\Anaconda\envs\test3\python.exe -m torch.distributed.run `
  --standalone `
  --nnodes=1 `
  --nproc_per_node=1 `
  -m verl.trainer.sft_trainer `
  data.train_files=D:\learnAI\verl\yhy\datasets\gsm8k_sft\train_500.parquet `
  data.val_files=D:\learnAI\verl\yhy\datasets\gsm8k_sft\test.parquet `
  data.messages_key=messages `
  data.train_batch_size=1 `
  data.micro_batch_size_per_gpu=1 `
  data.max_length=512 `
  data.truncation=right `
  model.path=D:\learnAI\verl\yhy\models\base\qwen3_0d6B `
  model.lora_rank=16 `
  model.lora_alpha=32 `
  model.target_modules=all-linear `
  model.use_remove_padding=True `
  optim.lr=1e-4 `
  trainer.default_local_dir=D:\learnAI\verl\yhy\models\sft\verl_qwen3_0d6b_gsm8k_lora `
  trainer.project_name=qwen3_0d6b_sft `
  trainer.experiment_name=gsm8k_lora_rtx4070 `
  trainer.total_epochs=1 `
  trainer.logger=console `
  trainer.save_freq=after_each_epoch `
  trainer.test_freq=after_each_epoch
```


## 模块 8. 结果记录模板

```text
checkpoint | exact match | format rate | avg length | 普通问答是否乱套 #### | 备注
base       | ...         | ...         | ...        | ...                  | ...
sft        | ...         | ...         | ...        | ...                  | ...
```

实验观察：

```text
现象:
可能原因:
检查方法:
下一步动作:
```
